# Lekcja 10 - Agenci AI w produkcji

W tej lekcji nauczysz się **wzorców produkcyjnych** dla agentów AI przy użyciu Microsoft Agent Framework (Python). Omawiamy:

- **Obserwowalność** — dodawanie pomiarów czasu i rejestrowania do interakcji agenta
- **Ewaluacja** — użycie agenta ewaluacyjnego do oceny jakości odpowiedzi
- **Zarządzanie kosztami** — strategie optymalizacji tokenów i wyboru modelu

Scenariusz to **agent podróży**, który pomaga użytkownikom planować wyjazdy, z warstwami monitorowania i ewaluacji.


## Konfiguracja


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Rozważania dotyczące produkcji

Przenoszenie agentów AI z prototypów do środowiska produkcyjnego wymaga starannego zwrócenia uwagi na trzy filary:

1. **Obserwowalność** — Musisz mieć wgląd w to, co robi agent, ile to zajmuje i których narzędzi używa. Bez śledzenia i logowania debugowanie problemów produkcyjnych jest praktycznie niemożliwe.

2. **Ewaluacja** — Automatyczne kontrole jakości zapewniają, że odpowiedzi agenta pozostają z czasem dokładne, kompletne i pomocne. Agent oceniający może punktować odpowiedzi na podstawie zdefiniowanych kryteriów.

3. **Zarządzanie kosztami** — Użycie tokenów bezpośrednio wpływa na koszty. Strategie takie jak optymalizacja promptów, wybór modelu i buforowanie pomagają utrzymać wydatki pod kontrolą bez utraty jakości.


## Tworzenie obserwowalnego agenta

Definiujemy narzędzia podróżne i opakowujemy wywołanie agenta pomiarem czasu, aby móc monitorować opóźnienia. W środowisku produkcyjnym należałoby zintegrować to z OpenTelemetry lub podobnym zapleczem do śledzenia.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Wzorce ewaluacji

Powszechnym wzorcem w produkcji jest użycie drugiego agenta jako **ewaluatora**. Ewaluator ocenia odpowiedź głównego agenta według wcześniej zdefiniowanych kryteriów, takich jak kompletność, dokładność i pomocność.

To umożliwia:
- Automatyczne bramki jakości zanim odpowiedzi trafią do użytkowników
- Wykrywanie regresji, gdy zmieniają się polecenia lub modele
- Ciągłe monitorowanie wydajności agenta w czasie


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategie zarządzania kosztami

Kontrola kosztów jest kluczowa dla produkcyjnych agentów AI. Oto kluczowe strategie:

| Strategia | Opis |
|---|---|
| **Optymalizacja promptów** | Utrzymuj instrukcje systemowe zwięzłe. Usuń zbędny kontekst, aby zmniejszyć liczbę tokenów wejściowych. |
| **Wybór modelu** | Używaj mniejszych, tańszych modeli (np. GPT-4o-mini) do prostych zadań, takich jak klasyfikacja czy ekstrakcja, a większe modele rezerwuj dla złożonego wnioskowania. |
| **Buforowanie** | Buforuj wyniki narzędzi i częste zapytania, aby uniknąć zbędnych wywołań API. |
| **Limity tokenów** | Ustal limity `max_tokens`, aby zapobiec niespodziewanie długim odpowiedziom. |
| **Grupowanie** | Grupuj wiele zapytań użytkownika w jednym wywołaniu API, jeśli to możliwe. |

W praktyce dobrze sprawdza się podejście warstwowe: kieruj proste żądania do szybkiego, niedrogiego modelu i eskaluj tylko złożone zapytania do bardziej zaawansowanego.


## Summary

W tej lekcji nauczyłeś się, jak:

1. **Dodawać obserwowalność** do interakcji agentów poprzez mierzenie czasu i logowanie, tworząc podstawy do śledzenia i monitorowania.
2. **Automatycznie oceniać odpowiedzi agenta** przy użyciu agenta-ewaluatora, który ocenia kompletność, dokładność i użyteczność.
3. **Zarządzać kosztami** poprzez optymalizację promptów, wybór modelu, buforowanie i budżety tokenów.

Te wzorce produkcyjne pomagają zapewnić, że Twoi agenci AI są niezawodni, mierzalni i efektywni kosztowo na dużą skalę.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Zastrzeżenie:
Niniejszy dokument został przetłumaczony przy użyciu usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dokładamy starań, aby zapewnić poprawność, pamiętaj, że tłumaczenia automatyczne mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym należy uznać za źródło nadrzędne. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za żadne nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
